In [ ]:
from pathlib import Path
import re

import pandas as pd
import matplotlib.pyplot as plt

RUNS_DIR = Path(
    "/blue/bala1s/krishnap.kalivel/TLFHydrodynamics/"
    "thinfilm-closure-ml/codes/ml-models/thinfilm_mlp/runs/sweep_01"
)

In [ ]:
pattern = re.compile(r"^(?P<group>.+)_j(?P<j>\d+)_s(?P<s>\d+)$")

histories = {}
records = []

for history_path in sorted(RUNS_DIR.glob("*/history.csv")):
    match = pattern.fullmatch(history_path.parent.name)
    if match is None:
        continue

    group = match["group"]
    j = int(match["j"])
    s = int(match["s"])

    history = pd.read_csv(history_path)

    required = {"epoch", "train_loss"}
    if not required.issubset(history.columns):
        raise ValueError(f"Missing required columns in {history_path}")

    history = history.sort_values("epoch").reset_index(drop=True)

    if history.empty:
        continue

    histories[(group, j, s)] = history

    records.append({
        "group": group,
        "j": j,
        "s": s,
        "last_epoch": int(history["epoch"].max()),
        "final_train_loss": history["train_loss"].iloc[-1],
    })

if not records:
    raise FileNotFoundError(f"No training histories found in {RUNS_DIR}")

model_table = (
    pd.DataFrame(records)
    .sort_values(["group", "j", "s"])
    .reset_index(drop=True)
)

# display(model_table)

In [ ]:
TARGET_GROUP = "corrections"   # "corrections" or "physical"

FIXED_PARAMETER = "j"          # Fix "j" and vary s, or fix "s" and vary j
FIXED_VALUE = 1

# None includes every available value of the varying parameter.
# Alternatively, specify a subset, such as [1, 3, 5].
VARYING_VALUES = None

LOG_Y = True
FIGSIZE = (8, 5)

In [ ]:
if FIXED_PARAMETER not in {"j", "s"}:
    raise ValueError("FIXED_PARAMETER must be 'j' or 's'.")

varying_parameter = "s" if FIXED_PARAMETER == "j" else "j"

selected = model_table.loc[
    (model_table["group"] == TARGET_GROUP)
    & (model_table[FIXED_PARAMETER] == FIXED_VALUE)
].copy()

if VARYING_VALUES is not None:
    selected = selected.loc[
        selected[varying_parameter].isin(VARYING_VALUES)
    ]

selected = selected.sort_values(varying_parameter)

if selected.empty:
    available = model_table.loc[
        model_table["group"] == TARGET_GROUP, ["j", "s"]
    ]
    raise ValueError(
        "No models match your selection.\n"
        f"Available combinations for {TARGET_GROUP}:\n"
        f"{available.to_string(index=False)}"
    )

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

for row in selected.itertuples(index=False):
    history = histories[(row.group, row.j, row.s)]
    varying_value = getattr(row, varying_parameter)
    label = rf"${varying_parameter}={varying_value}$"

    # Use the same color for each model in both panels.
    line, = axes[0].plot(
        history["epoch"],
        history["train_loss"],
        linewidth=1.8,
        label=label,
    )

    axes[1].plot(
        history["epoch"],
        history["val_loss"],
        linewidth=1.8,
        color=line.get_color(),
        label=label,
    )

for ax, title in zip(axes, ["Training", "Validation"]):
    if LOG_Y:
        ax.set_yscale("log")

    ax.set_xlabel("Epoch")
    ax.set_title(title)
    ax.legend(title="Varying parameter")
    ax.grid(True, which="both", alpha=0.25)

axes[0].set_ylabel("Loss (standardized MSE)")

fig.suptitle(
    rf"{TARGET_GROUP.capitalize()} models: "
    rf"${FIXED_PARAMETER}={FIXED_VALUE}$"
)

plt.show()

In [ ]:
TARGET_GROUP = "corrections"   # "corrections" or "physical"

FIXED_PARAMETER = "s"          # Fix "j" and vary s, or fix "s" and vary j
FIXED_VALUE = 3

# None includes every available value of the varying parameter.
# Alternatively, specify a subset, such as [1, 3, 5].
VARYING_VALUES = None

LOG_Y = True
FIGSIZE = (8, 5)

In [ ]:
if FIXED_PARAMETER not in {"j", "s"}:
    raise ValueError("FIXED_PARAMETER must be 'j' or 's'.")

varying_parameter = "s" if FIXED_PARAMETER == "j" else "j"

selected = model_table.loc[
    (model_table["group"] == TARGET_GROUP)
    & (model_table[FIXED_PARAMETER] == FIXED_VALUE)
].copy()

if VARYING_VALUES is not None:
    selected = selected.loc[
        selected[varying_parameter].isin(VARYING_VALUES)
    ]

selected = selected.sort_values(varying_parameter)

if selected.empty:
    available = model_table.loc[
        model_table["group"] == TARGET_GROUP, ["j", "s"]
    ]
    raise ValueError(
        "No models match your selection.\n"
        f"Available combinations for {TARGET_GROUP}:\n"
        f"{available.to_string(index=False)}"
    )

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

for row in selected.itertuples(index=False):
    history = histories[(row.group, row.j, row.s)]
    varying_value = getattr(row, varying_parameter)
    label = rf"${varying_parameter}={varying_value}$"

    # Use the same color for each model in both panels.
    line, = axes[0].plot(
        history["epoch"],
        history["train_loss"],
        linewidth=1.8,
        label=label,
    )

    axes[1].plot(
        history["epoch"],
        history["val_loss"],
        linewidth=1.8,
        color=line.get_color(),
        label=label,
    )

for ax, title in zip(axes, ["Training", "Validation"]):
    if LOG_Y:
        ax.set_yscale("log")

    ax.set_xlabel("Epoch")
    ax.set_title(title)
    ax.legend(title="Varying parameter")
    ax.grid(True, which="both", alpha=0.25)

axes[0].set_ylabel("Loss (standardized MSE)")

fig.suptitle(
    rf"{TARGET_GROUP.capitalize()} models: "
    rf"${FIXED_PARAMETER}={FIXED_VALUE}$"
)

plt.show()

In [ ]:
TARGET_GROUP = "corrections"  # Or "physical"
VARY_PARAMETER = "s"         # Parameter distinguishing the curves
CURVE_VALUES = None          # None = all; alternatively [1, 3, 5]
EPOCH = 100
LOG_Y = True

In [ ]:
if VARY_PARAMETER not in {"j", "s"}:
    raise ValueError("VARY_PARAMETER must be 'j' or 's'.")

x_parameter = "j" if VARY_PARAMETER == "s" else "s"

# Extract losses at exactly the requested epoch.
records = []

for (group, j, s), history in histories.items():
    if group != TARGET_GROUP:
        continue

    epoch_data = history.loc[history["epoch"] == EPOCH]

    if epoch_data.empty:
        print(f"Skipping {group}_j{j}_s{s}: epoch {EPOCH} missing.")
        continue

    row = epoch_data.iloc[-1]

    records.append({
        "j": j,
        "s": s,
        "train_loss": row["train_loss"],
        "val_loss": row["val_loss"],
    })

if not records:
    raise ValueError(f"No {TARGET_GROUP} histories contain epoch {EPOCH}.")

epoch_losses = pd.DataFrame(records)

if CURVE_VALUES is not None:
    epoch_losses = epoch_losses.loc[
        epoch_losses[VARY_PARAMETER].isin(CURVE_VALUES)
    ]

if epoch_losses.empty:
    raise ValueError("No models match the selected curve values.")

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5),
    sharex=True,
    sharey=False,
    constrained_layout=True,
)

for value, data in epoch_losses.groupby(VARY_PARAMETER, sort=True):
    data = data.sort_values(x_parameter)
    label = rf"${VARY_PARAMETER}={value}$"

    line, = axes[0].plot(
        data[x_parameter],
        data["train_loss"],
        marker="o",
        linewidth=1.8,
        label=label,
    )

    axes[1].plot(
        data[x_parameter],
        data["val_loss"],
        marker="o",
        linewidth=1.8,
        color=line.get_color(),
        label=label,
    )

x_labels = {
    "j": r"Half-window $j$",
    "s": r"Stride $s$",
}

for ax, title in zip(axes, ["Training", "Validation"]):
    if LOG_Y:
        ax.set_yscale("log")

    ax.set_xlabel(x_labels[x_parameter])
    ax.set_ylabel("Loss (standardized MSE)")
    ax.set_title(title)
    ax.set_xticks(sorted(epoch_losses[x_parameter].unique()))
    ax.legend(title=rf"Varying ${VARY_PARAMETER}$")
    ax.grid(True, which="both", alpha=0.25)

fig.suptitle(
    f"{TARGET_GROUP.capitalize()} models — epoch {EPOCH}"
)

plt.show()

In [ ]:
TARGET_GROUP = "physical"  # Or "physical"
VARY_PARAMETER = "s"         # Parameter distinguishing the curves
CURVE_VALUES = None          # None = all; alternatively [1, 3, 5]
EPOCH = 100
LOG_Y = True

In [ ]:
if VARY_PARAMETER not in {"j", "s"}:
    raise ValueError("VARY_PARAMETER must be 'j' or 's'.")

x_parameter = "j" if VARY_PARAMETER == "s" else "s"

# Extract losses at exactly the requested epoch.
records = []

for (group, j, s), history in histories.items():
    if group != TARGET_GROUP:
        continue

    epoch_data = history.loc[history["epoch"] == EPOCH]

    if epoch_data.empty:
        print(f"Skipping {group}_j{j}_s{s}: epoch {EPOCH} missing.")
        continue

    row = epoch_data.iloc[-1]

    records.append({
        "j": j,
        "s": s,
        "train_loss": row["train_loss"],
        "val_loss": row["val_loss"],
    })

if not records:
    raise ValueError(f"No {TARGET_GROUP} histories contain epoch {EPOCH}.")

epoch_losses = pd.DataFrame(records)

if CURVE_VALUES is not None:
    epoch_losses = epoch_losses.loc[
        epoch_losses[VARY_PARAMETER].isin(CURVE_VALUES)
    ]

if epoch_losses.empty:
    raise ValueError("No models match the selected curve values.")

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5),
    sharex=True,
    sharey=False,
    constrained_layout=True,
)

for value, data in epoch_losses.groupby(VARY_PARAMETER, sort=True):
    data = data.sort_values(x_parameter)
    label = rf"${VARY_PARAMETER}={value}$"

    line, = axes[0].plot(
        data[x_parameter],
        data["train_loss"],
        marker="o",
        linewidth=1.8,
        label=label,
    )

    axes[1].plot(
        data[x_parameter],
        data["val_loss"],
        marker="o",
        linewidth=1.8,
        color=line.get_color(),
        label=label,
    )

x_labels = {
    "j": r"Half-window $j$",
    "s": r"Stride $s$",
}

for ax, title in zip(axes, ["Training", "Validation"]):
    if LOG_Y:
        ax.set_yscale("log")

    ax.set_xlabel(x_labels[x_parameter])
    ax.set_ylabel("Loss (standardized MSE)")
    ax.set_title(title)
    ax.set_xticks(sorted(epoch_losses[x_parameter].unique()))
    ax.legend(title=rf"Varying ${VARY_PARAMETER}$")
    ax.grid(True, which="both", alpha=0.25)

fig.suptitle(
    f"{TARGET_GROUP.capitalize()} models — epoch {EPOCH}"
)

plt.show()

In [ ]:
TARGET_GROUP = "corrections"  # Or "physical"
VARY_PARAMETER = "j"         # Parameter distinguishing the curves
CURVE_VALUES = None          # None = all; alternatively [1, 3, 5]
EPOCH = 100
LOG_Y = True

In [ ]:
if VARY_PARAMETER not in {"j", "s"}:
    raise ValueError("VARY_PARAMETER must be 'j' or 's'.")

x_parameter = "j" if VARY_PARAMETER == "s" else "s"

# Extract losses at exactly the requested epoch.
records = []

for (group, j, s), history in histories.items():
    if group != TARGET_GROUP:
        continue

    epoch_data = history.loc[history["epoch"] == EPOCH]

    if epoch_data.empty:
        print(f"Skipping {group}_j{j}_s{s}: epoch {EPOCH} missing.")
        continue

    row = epoch_data.iloc[-1]

    records.append({
        "j": j,
        "s": s,
        "train_loss": row["train_loss"],
        "val_loss": row["val_loss"],
    })

if not records:
    raise ValueError(f"No {TARGET_GROUP} histories contain epoch {EPOCH}.")

epoch_losses = pd.DataFrame(records)

if CURVE_VALUES is not None:
    epoch_losses = epoch_losses.loc[
        epoch_losses[VARY_PARAMETER].isin(CURVE_VALUES)
    ]

if epoch_losses.empty:
    raise ValueError("No models match the selected curve values.")

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5),
    sharex=True,
    sharey=False,
    constrained_layout=True,
)

for value, data in epoch_losses.groupby(VARY_PARAMETER, sort=True):
    data = data.sort_values(x_parameter)
    label = rf"${VARY_PARAMETER}={value}$"

    line, = axes[0].plot(
        data[x_parameter],
        data["train_loss"],
        marker="o",
        linewidth=1.8,
        label=label,
    )

    axes[1].plot(
        data[x_parameter],
        data["val_loss"],
        marker="o",
        linewidth=1.8,
        color=line.get_color(),
        label=label,
    )

x_labels = {
    "j": r"Half-window $j$",
    "s": r"Stride $s$",
}

for ax, title in zip(axes, ["Training", "Validation"]):
    if LOG_Y:
        ax.set_yscale("log")

    ax.set_xlabel(x_labels[x_parameter])
    ax.set_ylabel("Loss (standardized MSE)")
    ax.set_title(title)
    ax.set_xticks(sorted(epoch_losses[x_parameter].unique()))
    ax.legend(title=rf"Varying ${VARY_PARAMETER}$")
    ax.grid(True, which="both", alpha=0.25)

fig.suptitle(
    f"{TARGET_GROUP.capitalize()} models — epoch {EPOCH}"
)

plt.show()

In [ ]:
TARGET_GROUP = "physical"  # Or "physical"
VARY_PARAMETER = "j"         # Parameter distinguishing the curves
CURVE_VALUES = None          # None = all; alternatively [1, 3, 5]
EPOCH = 100
LOG_Y = True

In [ ]:
if VARY_PARAMETER not in {"j", "s"}:
    raise ValueError("VARY_PARAMETER must be 'j' or 's'.")

x_parameter = "j" if VARY_PARAMETER == "s" else "s"

# Extract losses at exactly the requested epoch.
records = []

for (group, j, s), history in histories.items():
    if group != TARGET_GROUP:
        continue

    epoch_data = history.loc[history["epoch"] == EPOCH]

    if epoch_data.empty:
        print(f"Skipping {group}_j{j}_s{s}: epoch {EPOCH} missing.")
        continue

    row = epoch_data.iloc[-1]

    records.append({
        "j": j,
        "s": s,
        "train_loss": row["train_loss"],
        "val_loss": row["val_loss"],
    })

if not records:
    raise ValueError(f"No {TARGET_GROUP} histories contain epoch {EPOCH}.")

epoch_losses = pd.DataFrame(records)

if CURVE_VALUES is not None:
    epoch_losses = epoch_losses.loc[
        epoch_losses[VARY_PARAMETER].isin(CURVE_VALUES)
    ]

if epoch_losses.empty:
    raise ValueError("No models match the selected curve values.")

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5),
    sharex=True,
    sharey=False,
    constrained_layout=True,
)

for value, data in epoch_losses.groupby(VARY_PARAMETER, sort=True):
    data = data.sort_values(x_parameter)
    label = rf"${VARY_PARAMETER}={value}$"

    line, = axes[0].plot(
        data[x_parameter],
        data["train_loss"],
        marker="o",
        linewidth=1.8,
        label=label,
    )

    axes[1].plot(
        data[x_parameter],
        data["val_loss"],
        marker="o",
        linewidth=1.8,
        color=line.get_color(),
        label=label,
    )

x_labels = {
    "j": r"Half-window $j$",
    "s": r"Stride $s$",
}

for ax, title in zip(axes, ["Training", "Validation"]):
    if LOG_Y:
        ax.set_yscale("log")

    ax.set_xlabel(x_labels[x_parameter])
    ax.set_ylabel("Loss (standardized MSE)")
    ax.set_title(title)
    ax.set_xticks(sorted(epoch_losses[x_parameter].unique()))
    ax.legend(title=rf"Varying ${VARY_PARAMETER}$")
    ax.grid(True, which="both", alpha=0.25)

fig.suptitle(
    f"{TARGET_GROUP.capitalize()} models — epoch {EPOCH}"
)

plt.show()

Evaluation Metrics

In [ ]:
metric_tables = []

for metrics_path in sorted(RUNS_DIR.glob("*/eval_val/metrics.csv")):
    match = pattern.fullmatch(metrics_path.parent.parent.name)
    if match is None:
        continue

    table = pd.read_csv(metrics_path)
    table["group"] = match["group"]
    table["j"] = int(match["j"])
    table["s"] = int(match["s"])

    metric_tables.append(table)

if not metric_tables:
    raise FileNotFoundError(
        f"No eval_val/metrics.csv files found under {RUNS_DIR}"
    )

all_metrics = pd.concat(metric_tables, ignore_index=True)

# display(
#     all_metrics[["group", "target"]]
#     .drop_duplicates()
#     .sort_values(["group", "target"])
#     .reset_index(drop=True)
# )

In [ ]:
TARGET_GROUP = "physical"     # "physical" or "corrections"
OUTPUT_PARAMETER = "int_px_dy"

# "s": plot R² versus j, with one curve per s.
# "j": plot R² versus s, with one curve per j.
VARY_PARAMETER = "s"
CURVE_VALUES = None           # None = all; alternatively [1, 3, 5]

# Choose "CASE_AVERAGE", "POOLED", or an exact simulation name.
CASE = "CASE_AVERAGE"

if VARY_PARAMETER not in {"j", "s"}:
    raise ValueError("VARY_PARAMETER must be 'j' or 's'.")

x_parameter = "j" if VARY_PARAMETER == "s" else "s"

selected = all_metrics.loc[
    (all_metrics["group"] == TARGET_GROUP)
    & (all_metrics["target"] == OUTPUT_PARAMETER)
    & (all_metrics["case"] == CASE)
].copy()

if CURVE_VALUES is not None:
    selected = selected.loc[
        selected[VARY_PARAMETER].isin(CURVE_VALUES)
    ]

if selected.empty:
    raise ValueError(
        "No metrics match the selected group, output, case, and curve values."
    )

if selected.duplicated(["j", "s"]).any():
    raise ValueError("Multiple metric rows found for the same j, s combination.")

fig, ax = plt.subplots(
    figsize=(8, 5),
    constrained_layout=True,
)

for value, data in selected.groupby(VARY_PARAMETER, sort=True):
    data = data.sort_values(x_parameter)

    ax.plot(
        data[x_parameter],
        data["R2"],
        marker="o",
        linewidth=1.8,
        label=rf"${VARY_PARAMETER}={value}$",
    )

ax.set_xlabel(
    r"Half-window $j$" if x_parameter == "j" else r"Stride $s$"
)
ax.set_ylabel(r"$R^2$")
ax.set_xticks(sorted(selected[x_parameter].unique()))

ax.set_title(
    f"{TARGET_GROUP.capitalize()} — {OUTPUT_PARAMETER}\n"
    f"Validation: {CASE}"
)

ax.legend(title=rf"Varying ${VARY_PARAMETER}$")
ax.grid(True, alpha=0.25)

plt.show()

In [ ]:
# Spatial half-width, measured in DNS grid intervals.
selected["half_width_dx"] = selected["j"] * selected["s"]

fig, ax = plt.subplots(
    figsize=(8, 5),
    constrained_layout=True,
)

for value, data in selected.groupby(VARY_PARAMETER, sort=True):
    data = data.sort_values("half_width_dx")

    ax.plot(
        data["half_width_dx"],
        data["R2"],
        marker="o",
        linewidth=1.8,
        label=rf"${VARY_PARAMETER}={value}$",
    )

ax.set_xlabel(r"Spatial half-width $R/\Delta x_{\mathrm{DNS}} = js$")
ax.set_ylabel(r"$R^2$")

ax.set_title(
    f"{TARGET_GROUP.capitalize()} — {OUTPUT_PARAMETER}\n"
    f"Validation: {CASE}"
)

ax.legend(title=rf"Varying ${VARY_PARAMETER}$")
ax.grid(True, alpha=0.25)

plt.show()

In [ ]:
DX = 0.0488
selected["half_width"] = selected["j"] * selected["s"] * DX

fig, ax = plt.subplots(
    figsize=(8, 5),
    constrained_layout=True,
)

for value, data in selected.groupby(VARY_PARAMETER, sort=True):
    data = data.sort_values("half_width")

    ax.plot(
        data["half_width"],
        data["R2"],
        marker="o",
        linewidth=1.8,
        label=rf"${VARY_PARAMETER}={value}$",
    )

ax.set_xlabel(r"Nondimensional spatial half-width $R = js\Delta x$")
ax.set_ylabel(r"$R^2$")

ax.set_title(
    f"{TARGET_GROUP.capitalize()} — {OUTPUT_PARAMETER}\n"
    f"Validation: {CASE}"
)

ax.legend(title=rf"Varying ${VARY_PARAMETER}$")
ax.grid(True, alpha=0.25)

plt.show()

In [ ]:
TARGET_GROUP = "corrections"     # "physical" or "corrections"
OUTPUT_PARAMETER = "delta_Px_ML"

# "s": plot R² versus j, with one curve per s.
# "j": plot R² versus s, with one curve per j.
VARY_PARAMETER = "s"
CURVE_VALUES = None           # None = all; alternatively [1, 3, 5]

# Choose "CASE_AVERAGE", "POOLED", or an exact simulation name.
CASE = "CASE_AVERAGE"

if VARY_PARAMETER not in {"j", "s"}:
    raise ValueError("VARY_PARAMETER must be 'j' or 's'.")

x_parameter = "j" if VARY_PARAMETER == "s" else "s"

selected = all_metrics.loc[
    (all_metrics["group"] == TARGET_GROUP)
    & (all_metrics["target"] == OUTPUT_PARAMETER)
    & (all_metrics["case"] == CASE)
].copy()

if CURVE_VALUES is not None:
    selected = selected.loc[
        selected[VARY_PARAMETER].isin(CURVE_VALUES)
    ]

if selected.empty:
    raise ValueError(
        "No metrics match the selected group, output, case, and curve values."
    )

if selected.duplicated(["j", "s"]).any():
    raise ValueError("Multiple metric rows found for the same j, s combination.")

fig, ax = plt.subplots(
    figsize=(8, 5),
    constrained_layout=True,
)

for value, data in selected.groupby(VARY_PARAMETER, sort=True):
    data = data.sort_values(x_parameter)

    ax.plot(
        data[x_parameter],
        data["R2"],
        marker="o",
        linewidth=1.8,
        label=rf"${VARY_PARAMETER}={value}$",
    )

ax.set_xlabel(
    r"Half-window $j$" if x_parameter == "j" else r"Stride $s$"
)
ax.set_ylabel(r"$R^2$")
# ax.set_ylabel(r"$RMSE$")
ax.set_xticks(sorted(selected[x_parameter].unique()))
# ax.set_xlim(left=1)
ax.set_title(
    f"{TARGET_GROUP.capitalize()} — {OUTPUT_PARAMETER}\n"
    f"Validation: {CASE}"
)

ax.legend(title=rf"Varying ${VARY_PARAMETER}$")
ax.grid(True, alpha=0.25)

plt.show()

In [ ]:
# Spatial half-width, measured in DNS grid intervals.
selected["half_width_dx"] = selected["j"] * selected["s"]

fig, ax = plt.subplots(
    figsize=(8, 5),
    constrained_layout=True,
)

for value, data in selected.groupby(VARY_PARAMETER, sort=True):
    data = data.sort_values("half_width_dx")

    ax.plot(
        data["half_width_dx"],
        data["R2"],
        marker="o",
        linewidth=1.8,
        label=rf"${VARY_PARAMETER}={value}$",
    )

ax.set_xlabel(r"Spatial half-width $R/\Delta x_{\mathrm{DNS}} = js$")
ax.set_ylabel(r"$R^2$")

ax.set_title(
    f"{TARGET_GROUP.capitalize()} — {OUTPUT_PARAMETER}\n"
    f"Validation: {CASE}"
)

ax.legend(title=rf"Varying ${VARY_PARAMETER}$")
ax.grid(True, alpha=0.25)

plt.show()

In [ ]:
DX = 0.0488
selected["half_width"] = selected["j"] * selected["s"] * DX

fig, ax = plt.subplots(
    figsize=(8, 5),
    constrained_layout=True,
)

for value, data in selected.groupby(VARY_PARAMETER, sort=True):
    data = data.sort_values("half_width")

    ax.plot(
        data["half_width"],
        data["R2"],
        marker="o",
        linewidth=1.8,
        label=rf"${VARY_PARAMETER}={value}$",
    )

ax.set_xlabel(r"Nondimensional spatial half-width $R = js\Delta x$")
ax.set_ylabel(r"$R^2$")

ax.set_title(
    f"{TARGET_GROUP.capitalize()} — {OUTPUT_PARAMETER}\n"
    f"Validation: {CASE}"
)

ax.legend(title=rf"Varying ${VARY_PARAMETER}$")
ax.grid(True, alpha=0.25)

plt.show()

In [ ]:
TARGET_GROUP = "corrections"     # "physical" or "corrections"
OUTPUT_PARAMETER = "delta_Mx_ML"

# "s": plot R² versus j, with one curve per s.
# "j": plot R² versus s, with one curve per j.
VARY_PARAMETER = "s"
CURVE_VALUES = None           # None = all; alternatively [1, 3, 5]

# Choose "CASE_AVERAGE", "POOLED", or an exact simulation name.
CASE = "CASE_AVERAGE"

if VARY_PARAMETER not in {"j", "s"}:
    raise ValueError("VARY_PARAMETER must be 'j' or 's'.")

x_parameter = "j" if VARY_PARAMETER == "s" else "s"

selected = all_metrics.loc[
    (all_metrics["group"] == TARGET_GROUP)
    & (all_metrics["target"] == OUTPUT_PARAMETER)
    & (all_metrics["case"] == CASE)
].copy()

if CURVE_VALUES is not None:
    selected = selected.loc[
        selected[VARY_PARAMETER].isin(CURVE_VALUES)
    ]

if selected.empty:
    raise ValueError(
        "No metrics match the selected group, output, case, and curve values."
    )

if selected.duplicated(["j", "s"]).any():
    raise ValueError("Multiple metric rows found for the same j, s combination.")

fig, ax = plt.subplots(
    figsize=(8, 5),
    constrained_layout=True,
)

for value, data in selected.groupby(VARY_PARAMETER, sort=True):
    data = data.sort_values(x_parameter)

    ax.plot(
        data[x_parameter],
        data["R2"],
        marker="o",
        linewidth=1.8,
        label=rf"${VARY_PARAMETER}={value}$",
    )

ax.set_xlabel(
    r"Half-window $j$" if x_parameter == "j" else r"Stride $s$"
)
ax.set_ylabel(r"$R^2$")
ax.set_xticks(sorted(selected[x_parameter].unique()))

ax.set_title(
    f"{TARGET_GROUP.capitalize()} — {OUTPUT_PARAMETER}\n"
    f"Validation: {CASE}"
)

ax.legend(title=rf"Varying ${VARY_PARAMETER}$")
ax.grid(True, alpha=0.25)

plt.show()

In [ ]:
# Spatial half-width, measured in DNS grid intervals.
selected["half_width_dx"] = selected["j"] * selected["s"]

fig, ax = plt.subplots(
    figsize=(8, 5),
    constrained_layout=True,
)

for value, data in selected.groupby(VARY_PARAMETER, sort=True):
    data = data.sort_values("half_width_dx")

    ax.plot(
        data["half_width_dx"],
        data["R2"],
        marker="o",
        linewidth=1.8,
        label=rf"${VARY_PARAMETER}={value}$",
    )

ax.set_xlabel(r"Spatial half-width $R/\Delta x_{\mathrm{DNS}} = js$")
ax.set_ylabel(r"$R^2$")

ax.set_title(
    f"{TARGET_GROUP.capitalize()} — {OUTPUT_PARAMETER}\n"
    f"Validation: {CASE}"
)

ax.legend(title=rf"Varying ${VARY_PARAMETER}$")
ax.grid(True, alpha=0.25)

plt.show()

In [ ]:
DX = 0.0488
selected["half_width"] = selected["j"] * selected["s"] * DX

fig, ax = plt.subplots(
    figsize=(8, 5),
    constrained_layout=True,
)

for value, data in selected.groupby(VARY_PARAMETER, sort=True):
    data = data.sort_values("half_width")

    ax.plot(
        data["half_width"],
        data["R2"],
        marker="o",
        linewidth=1.8,
        label=rf"${VARY_PARAMETER}={value}$",
    )

ax.set_xlabel(r"Nondimensional spatial half-width $R = js\Delta x$")
ax.set_ylabel(r"$R^2$")

ax.set_title(
    f"{TARGET_GROUP.capitalize()} — {OUTPUT_PARAMETER}\n"
    f"Validation: {CASE}"
)

ax.legend(title=rf"Varying ${VARY_PARAMETER}$")
ax.grid(True, alpha=0.25)

plt.show()

Let's load j = 6, s = 1 Corrections file and see how spatial evolution looks for an heldout case

In [ ]:
from pathlib import Path
import sys

import h5py
import numpy as np
import torch
import matplotlib.pyplot as plt

CODE_DIR = Path(
    "/blue/bala1s/krishnap.kalivel/TLFHydrodynamics/"
    "thinfilm-closure-ml/codes/ml-models/thinfilm_mlp"
)

CHECKPOINT_PATH = (
    CODE_DIR / "runs/sweep_01/corrections_j6_s1/best.pt"
)

# Replace with the actual path to your chosen original HDF5 file.
DATA_FILE = Path(
    "/blue/bala1s/krishnap.kalivel/TLFHydrodynamics/thinfilm-closure-ml/datasets/BasC/param_space/ClosureData/"
    "thinfilm_closure_data_Re080_Ca0_010_theta90.h5"
)

TIME = 1800.0          # Nearest saved time will be selected
XLIM = (0.0, 800.0)   # Nondimensional x interval; None for entire domain

BATCH_SIZE = 8192
DEVICE = "cpu"       # Use "cuda" inside a GPU allocation

# None plots all outputs learned by this model.
PLOT_TARGETS = ["delta_Mx_ML", "delta_Tw_ML", "delta_Px_ML"]
# Example:
# PLOT_TARGETS = ["delta_Mx_ML", "delta_Tw_ML"]

In [ ]:
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from model import PhysicalMLP

device = torch.device(DEVICE)

# Load only checkpoints you trust.
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

model = PhysicalMLP(
    checkpoint["architecture"],
    checkpoint["normalization"],
).to(device)

model.load_state_dict(checkpoint["model"])
model.eval()

settings = checkpoint["args"]
j = int(settings["half_window"])
s = int(settings["stride"])
target_names = list(settings["targets"])

offsets = np.arange(-j, j + 1, dtype=int) * s

if (j, s) != (6, 1):
    raise ValueError(f"Expected j=6, s=1, but checkpoint has j={j}, s={s}.")

def read_parameter(h5_file, name):
    group = h5_file["parameters"]
    value = group[name][()] if name in group else group.attrs[name]
    return float(np.asarray(value).squeeze())

with h5py.File(DATA_FILE, "r") as f:
    x = np.asarray(f["x"][:], dtype=float).ravel()
    t = np.asarray(f["t"][:], dtype=float).ravel()

    if not np.all(np.diff(x) > 0):
        raise ValueError("x must be strictly increasing.")

    time_index = int(np.argmin(np.abs(t - TIME)))
    actual_time = float(t[time_index])

    def read_snapshot(name):
        if f[name].shape != (len(x), len(t)):
            raise ValueError(
                f"{name}: expected shape {(len(x), len(t))}, "
                f"found {f[name].shape}"
            )
        return np.asarray(f[name][:, time_index], dtype=np.float32)

    h = read_snapshot("h")
    q = read_snapshot("q")

    dns = np.column_stack([
        read_snapshot(name) for name in target_names
    ])

    global_inputs = np.array([
        read_parameter(f, "Re"),
        read_parameter(f, "Ca"),
        read_parameter(f, "angle"),  # Already in radians
    ], dtype=np.float32)

# Centers must have a complete stencil inside the original domain.
margin = j * s
centers = np.arange(margin, len(x) - margin)

if XLIM is not None:
    if XLIM[0] >= XLIM[1]:
        raise ValueError("XLIM must have increasing bounds.")
    centers = centers[
        (x[centers] >= XLIM[0]) & (x[centers] <= XLIM[1])
    ]

if centers.size == 0:
    raise ValueError("No complete input windows inside the selected x interval.")

neighbors = centers[:, None] + offsets[None, :]

h_window = h[neighbors]
q_window = q[neighbors]
dx_window = x[neighbors] - x[centers, None]

# Exact training order:
# [h_left, q_left, dx_left, ..., h_right, q_right, dx_right,
#  Re, Ca, angle]
window_inputs = np.stack(
    [h_window, q_window, dx_window],
    axis=-1,
).reshape(len(centers), -1)

X = np.concatenate([
    window_inputs,
    np.tile(global_inputs, (len(centers), 1)),
], axis=1).astype(np.float32)

valid_inputs = (
    np.isfinite(X).all(axis=1)
    & (h_window > 0).all(axis=1)
)

print(f"Checkpoint epoch: {checkpoint['epoch']}")
print(f"Model: j={j}, s={s}")
print(f"Outputs: {target_names}")
print(f"Requested time: {TIME:g}; selected time: {actual_time:g}")
print(f"Snapshot index: {time_index}")
print(f"Valid prediction centers: {valid_inputs.sum()} / {len(centers)}")

In [ ]:


predictions = np.full(
    (len(centers), len(target_names)),
    np.nan,
    dtype=np.float32,
)

valid_rows = np.flatnonzero(valid_inputs)

with torch.no_grad():
    for start in range(0, len(valid_rows), BATCH_SIZE):
        rows = valid_rows[start:start + BATCH_SIZE]

        inputs = torch.from_numpy(X[rows]).to(device)

        # forward() returns original nondimensional target values.
        predictions[rows] = model(inputs).cpu().numpy()

plot_targets = target_names if PLOT_TARGETS is None else PLOT_TARGETS

if not plot_targets:
    raise ValueError("Select at least one target.")

missing = set(plot_targets) - set(target_names)
if missing:
    raise ValueError(f"Targets not predicted by this checkpoint: {sorted(missing)}")

labels = {
    "delta_Dx_ML": r"$\Delta\mathcal{D}_{x,\mathrm{ML}}$",
    "delta_Mx_ML": r"$\Delta\mathcal{M}_{x,\mathrm{ML}}$",
    "delta_Px_ML": r"$\Delta\mathcal{P}_{x,\mathrm{ML}}$",
    "delta_Tw_ML": r"$\Delta\mathcal{T}_{w,\mathrm{ML}}$",
}

fig, axes = plt.subplots(
    len(plot_targets),
    1,
    figsize=(11, 3 * len(plot_targets)),
    sharex=True,
    squeeze=False,
    constrained_layout=True,
)

x_plot = x[centers]

for ax, name in zip(axes[:, 0], plot_targets):
    column = target_names.index(name)

    truth = dns[centers, column].copy()
    truth[~np.isfinite(truth)] = np.nan

    ax.plot(
        x_plot, truth,
        color="black",
        linewidth=1.8,
        label="DNS",
    )
    ax.plot(
        x_plot, predictions[:, column],
        color="tab:red",
        linestyle="--",
        linewidth=1.6,
        label="MLP",
    )

    ax.set_ylabel(labels.get(name, name))
    ax.grid(True, alpha=0.25)
    ax.legend()
    ax.set_xlim(500,600)
    # if XLIM is not None:
    #     ax.set_xlim(*XLIM)

axes[-1, 0].set_xlabel(r"Nondimensional position $x$")

fig.suptitle(
    f"{DATA_FILE.stem}\n"
    rf"$t={actual_time:g}$, $j={j}$, $s={s}$"
    f" — checkpoint epoch {checkpoint['epoch']}"
)

plt.show()